# Pairs Trading: Textbook & Reference

**Instructions:** Read through this notebook and take notes. It contains the complete theory and fully functional code for the pairs trading pipeline.

## Section 0: Data Download

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss
from itertools import combinations

# 10 Stock Universe grouped by Industry
industry_map = {
    'Tech': ['AAPL', 'MSFT', 'GOOGL', 'META'],
    'Financials': ['JPM', 'BAC', 'WFC'],
    'Energy': ['XOM', 'CVX', 'COP']
}
tickers = [ticker for group in industry_map.values() for ticker in group]

print("Downloading data...")
data = yf.download(tickers, start="2021-01-01", end="2024-01-01")['Close']
data = data.dropna()
print("Data downloaded.")


## Section 1: Stationarity & Unit Root Tests

### 📝 Answer: The ADF Test
The **Augmented Dickey-Fuller (ADF) test** checks whether a time series has a unit root, which would mean it is non-stationary.
*   **Null Hypothesis (H0):** The series has a unit root (it is NON-STATIONARY).
*   **Alternative Hypothesis (H1):** The series is stationary.
If the p-value is < 0.05, we reject the null hypothesis and assume the series is stationary.

### 📝 Answer: The KPSS Test
The **KPSS test** is another unit root test, but its hypotheses are flipped compared to the ADF test.
*   **Null Hypothesis (H0):** The series is STATIONARY around a deterministic trend.
*   **Alternative Hypothesis (H1):** The series has a unit root (non-stationary).
If the p-value is < 0.05, we reject the null hypothesis, meaning the series is non-stationary. Using both tests provides a robust check for stationarity.

In [ ]:
# Extract AAPL prices
aapl_prices = data['AAPL']

# Run the ADF test
adf_result = adfuller(aapl_prices)
print("ADF p-value:", adf_result[1])

# Run the KPSS test
import warnings
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    kpss_result = kpss(aapl_prices)
print("KPSS p-value:", kpss_result[1])


## Section 2: Pair Selection (Industry-Based)

In [ ]:
industry_pairs = []

# Iterate through `industry_map` and use combinations to create pairs
for industry, group_tickers in industry_map.items():
    # Generate pairs for this group
    pairs = list(combinations(group_tickers, 2))
    industry_pairs.extend(pairs)

print(f"Total intra-industry pairs generated: {len(industry_pairs)}")
print(industry_pairs[:5])


## Section 3: Spread Construction & Hedge Ratio

### 📝 Answer: Log Prices
We use **log prices** because changes in logarithmic prices approximate percentage returns. By modeling the spread of log prices ($Spread = \ln(A) - eta \ln(B)$), the spread represents the relative percentage difference between the two assets, making the strategy invariant to the absolute nominal price levels of the stocks. It also helps stabilize the variance over time.

In [ ]:
# Calculate log prices for AAPL and MSFT
log_aapl = np.log(data['AAPL'])
log_msft = np.log(data['MSFT'])

# Calculate Hedge Ratio using OLS (Y = AAPL, X = MSFT)
X = sm.add_constant(log_msft)
model = sm.OLS(log_aapl, X).fit()
hedge_ratio = model.params.iloc[1]

print(f"Hedge Ratio: {hedge_ratio}")

# Calculate the spread
spread = log_aapl - (hedge_ratio * log_msft)
spread.plot(title="AAPL - MSFT Log Spread")
plt.show()


## Section 4: Mean Reversion & Half-Life

### 📝 Answer: Half-Life
The **half-life of mean reversion** tells us the average time it takes for a spread to revert halfway back to its historical mean after deviating from it. 
It is derived by fitting an **Ornstein-Uhlenbeck (OU) process** to the spread, regressing the change in the spread ($\Delta y_t$) against its lagged value ($y_{t-1}$). The slope coefficient ($\lambda$) represents the speed of mean reversion. We calculate the half-life as `-ln(2) / lambda`.

In [ ]:
# Calculate lagged spread and delta spread
spread_lag = spread.shift(1)
spread_lag = spread_lag.dropna()
delta_spread = spread - spread_lag
delta_spread = delta_spread.dropna()

# Ensure alignment
spread_lag, delta_spread = spread_lag.align(delta_spread, join='inner')

# Run OLS to find lambda (slope)
X_hl = spread_lag
model_hl = sm.OLS(delta_spread, X_hl).fit()
lambda_val = model_hl.params.iloc[0]

# Calculate half-life
half_life = -np.log(2) / lambda_val
print(f"Half-life: {half_life:.2f} days")


## Section 5: Signal Generation with Bollinger Bands

### 📝 Answer: Bollinger Bands
**Bollinger Bands** consist of a rolling simple moving average (SMA) and two outer bands representing $+/- Z$ standard deviations away from the SMA. 
*   **Entry Signals:** If the spread is stationary, we expect it to revert to the mean. Therefore, we **Go Long** when the spread drops below the lower band (it is undervalued), and we **Go Short** when it spikes above the upper band (it is overvalued).
*   **Exit Signal:** We exit the position when the spread crosses the moving average (mean).

In [ ]:
# Define lookback window
window = 20

# Calculate rolling mean and standard deviation
rolling_mean = spread.rolling(window=window).mean()
rolling_std = spread.rolling(window=window).std()

# Construct Upper and Lower Bands (2 standard deviations)
upper_band = rolling_mean + (2 * rolling_std)
lower_band = rolling_mean - (2 * rolling_std)

# Generate Signals
signals = pd.Series(0, index=spread.index)
signals[spread < lower_band] = 1
signals[spread > upper_band] = -1

# Plot
plt.figure(figsize=(10,6))
plt.plot(spread, label='Spread')
plt.plot(rolling_mean, label='Moving Average', color='black', linestyle='--')
plt.plot(upper_band, label='Upper Band', color='red', linestyle=':')
plt.plot(lower_band, label='Lower Band', color='green', linestyle=':')
plt.legend()
plt.title("Bollinger Bands for AAPL/MSFT Spread")
plt.show()
